# mini-meta-harness — explore a run

Loads a completed run from `runs/` and tells the before/after story:
accuracy trend, cost trend, what the best harness ended up looking like,
and a confusion matrix comparing baseline vs. best.

In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve() if Path('runs').exists() is False else Path('.').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from mini_meta_harness.types import RunSummary, EvalExample
from mini_meta_harness import cost as cost_mod

# Pick the most recent run, or set RUN_ID = 'YYYY-MM-DD_HH-MM-SS'.
RUN_ID = None
runs_dir = ROOT / 'runs'
candidates = sorted([p for p in runs_dir.iterdir() if p.is_dir() and (p / 'summary.json').exists()])
if RUN_ID is None:
    run_dir = candidates[-1]
else:
    run_dir = runs_dir / RUN_ID
print('Loading', run_dir)
summary = RunSummary.model_validate(json.loads((run_dir / 'summary.json').read_text()))

## Config

In [ ]:
cfg = summary.config
pd.Series({
    'run_id': cfg.run_id,
    'proposer_model': cfg.proposer_model,
    'target_model': cfg.target_model,
    'iterations': cfg.iterations,
    'dataset': cfg.dataset,
    'eval_split_size': cfg.eval_split_size,
    'mock': cfg.mock,
    'schema_version': cfg.schema_version,
    'best_iteration_index': summary.best_iteration_index,
    'total_usd': summary.total_cost.estimated_usd,
}).to_frame('value')

## Accuracy over iterations

In [ ]:
score_df = pd.DataFrame([{
    'iteration': it.index,
    'accuracy': it.score.accuracy,
    'n_errors': it.score.n_errors,
    'mean_latency_ms': it.score.mean_latency_ms,
    'target_input_tokens': it.score.total_input_tokens,
    'target_output_tokens': it.score.total_output_tokens,
} for it in summary.iterations]).sort_values('iteration')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(score_df['iteration'], score_df['accuracy'], marker='o')
best = summary.best_iteration_index
ax.axvline(best, linestyle='--', alpha=0.4, label=f'best = iter {best}')
ax.set_xlabel('iteration'); ax.set_ylabel('accuracy'); ax.set_ylim(0, 1)
ax.set_title('Accuracy per iteration'); ax.legend(); ax.grid(alpha=0.3)
plt.show()
score_df

## Cost over iterations (proposer vs target)

In [ ]:
rows = []
for it in summary.iterations:
    # Per-iteration cost is re-derived from token counts in the trace plus
    # any proposer-input/output tokens we captured in filesystem_reads.
    target_in = it.score.total_input_tokens
    target_out = it.score.total_output_tokens
    target_usd = cost_mod.usd_for(cfg.target_model, target_in, target_out)
    # Proposer tokens aren't persisted per-iteration in v0.1.0; approximate
    # by dividing the running total evenly across non-baseline iterations.
    rows.append({'iteration': it.index, 'target_usd': target_usd})
cost_df = pd.DataFrame(rows).set_index('iteration')
proposer_total = max(summary.total_cost.estimated_usd - cost_df['target_usd'].sum(), 0)
non_baseline = [i for i in cost_df.index if i != 0]
cost_df['proposer_usd'] = 0.0
if non_baseline:
    cost_df.loc[non_baseline, 'proposer_usd'] = proposer_total / len(non_baseline)
cost_df[['proposer_usd', 'target_usd']].plot.bar(stacked=True, figsize=(8, 4))
plt.ylabel('USD'); plt.title('Per-iteration cost'); plt.tight_layout(); plt.show()
cost_df.round(4)

## Best iteration — code and reasoning

In [ ]:
best_it = next(it for it in summary.iterations if it.index == summary.best_iteration_index)
print(f'Best iteration: {best_it.index} — accuracy {best_it.score.accuracy:.3f}')
print('\n--- reasoning ---\n')
print(best_it.reasoning)
print('\n--- harness.py ---\n')
print(best_it.harness_code)

## Confusion matrix — baseline (iter 0) vs best

In [ ]:
def _load_traces(i: int) -> pd.DataFrame:
    path = run_dir / 'iterations' / f'{i:03d}' / 'eval_trace.jsonl'
    traces = [EvalExample.model_validate(json.loads(l)) for l in path.read_text().splitlines() if l.strip()]
    return pd.DataFrame([t.model_dump() for t in traces])

base_df = _load_traces(0)
best_df = _load_traces(summary.best_iteration_index)

labels = sorted(set(base_df['gold_label']).union(base_df['predicted_label'].dropna()).union(best_df['predicted_label'].dropna()))
def _cm(df):
    df = df.copy()
    df['predicted_label'] = df['predicted_label'].fillna('<error>')
    return pd.crosstab(df['gold_label'], df['predicted_label'], dropna=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(_cm(base_df), ax=axes[0], cmap='Blues', cbar=False, linewidths=0.2)
axes[0].set_title(f'Baseline (iter 0) — acc {base_df["correct"].mean():.3f}')
sns.heatmap(_cm(best_df), ax=axes[1], cmap='Greens', cbar=False, linewidths=0.2)
axes[1].set_title(f'Best (iter {summary.best_iteration_index}) — acc {best_df["correct"].mean():.3f}')
for ax in axes:
    ax.set_xlabel('predicted'); ax.set_ylabel('gold')
    plt.setp(ax.get_xticklabels(), rotation=70, ha='right')
plt.tight_layout(); plt.show()

## Proposer behavior per iteration

In [ ]:
behavior = pd.DataFrame([{
    'iteration': it.index,
    'accuracy': it.score.accuracy,
    'filesystem_reads': len(it.filesystem_reads),
    'unique_files_read': len({r.path for r in it.filesystem_reads}),
    'n_errors': it.score.n_errors,
    'mean_latency_ms': it.score.mean_latency_ms,
} for it in summary.iterations]).set_index('iteration')
behavior